# Giai đoạn 1: Tiền xử lý dữ liệu

## Mô tả
Notebook này thực hiện tiền xử lý dữ liệu UIT-VSFC theo quy trình:

```
┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│  Làm sạch văn bản │───▶│Chuẩn hóa teencode│───▶│  Tokenization   │
└────────┬─────────┘    └────────┬─────────┘    └────────┬─────────┘
         │                       │                       │
         ▼                       ▼                       ▼
  • Lowercase           • ko → không          • PhoBERT BPE
  • Remove special      • j → gì              • Subword tokenization
    characters          • cx → cũng           • Max length: 256
  • Normalize           • dc → được
    whitespace          • 100+ mappings
```

**Input:** `data/raw/` — Dữ liệu gốc UIT-VSFC

**Output:** `data/processed/` — Dữ liệu đã tiền xử lý

## 1. Setup và Import Libraries

In [1]:
# Chạy trên Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import numpy as np
from typing import List, Tuple, Dict
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [3]:
def set_seed(seed=42):
    np.random.seed(seed)

set_seed(42)

## 2. Configuration

In [4]:
class Config:
    # Paths - Chỉnh sửa khi chạy local
    BASE_DIR = '/content/drive/MyDrive/Student-Feedback-Sentiment-Analysis'

    # Input: Raw data
    RAW_DATA_DIR = f'{BASE_DIR}/data/raw'

    # Output: Processed data
    PROCESSED_DATA_DIR = f'{BASE_DIR}/data/processed'

    # Splits
    SPLITS = ['train', 'validation', 'test']

    # Label mapping
    NUM_CLASSES = 3
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

config = Config()
print(f'Raw data directory: {config.RAW_DATA_DIR}')
print(f'Output directory: {config.PROCESSED_DATA_DIR}')

Raw data directory: /content/drive/MyDrive/Student-Feedback-Sentiment-Analysis/data/raw
Output directory: /content/drive/MyDrive/Student-Feedback-Sentiment-Analysis/data/processed


## 3. Định nghĩa các hàm tiền xử lý

### 3.1. Teencode Dictionary

Mapping từ viết tắt/tiếng lóng mạng → từ chuẩn tiếng Việt

In [5]:
# Teencode Dictionary - Vietnamese Internet Slang
TEENCODE_DICT = {
    # Phủ định
    'ko': 'không', 'k': 'không', 'khong': 'không', 'hok': 'không', 'hk': 'không',
    'kg': 'không', 'k0': 'không', 'kô': 'không', 'hông': 'không',

    # Đại từ
    'j': 'gì', 'gj': 'gì', 'gi': 'gì',
    'm': 'mày', 'mik': 'mình', 'mk': 'mình', 'mìh': 'mình',
    't': 'tao', 'tui': 'tôi', 'tôi': 'tôi',
    'bạn': 'bạn', 'bn': 'bạn',

    # Phó từ
    'r': 'rồi', 'rùi': 'rồi', 'roi': 'rồi', 'ròi': 'rồi',
    'nữa': 'nữa', 'nua': 'nữa',
    'h': 'hơn', 'hn': 'hơn',

    # Thì, mà, nhưng
    'nhưng': 'nhưng', 'nhg': 'nhưng', 'nhung': 'nhưng', 'nhgư': 'nhưng',
    'thì': 'thì', 'thj': 'thì', 'thik': 'thích',
    'mà': 'mà', 'ma': 'mà',

    # Tần suất
    'cx': 'cũng', 'cũng': 'cũng', 'cug': 'cũng', 'cũg': 'cũng',
    'đc': 'được', 'dc': 'được', 'đươc': 'được', 'duoc': 'được',
    'bnh': 'bình', 'bth': 'bình thường', 'bthg': 'bình thường',

    # Trạng từ chỉ mức độ
    'lm': 'lắm', 'lém': 'lắm', 'lem': 'lắm', 'nw': 'nữa',
    'qá': 'quá', 'qa': 'quá', 'quá': 'quá', 'wa': 'quá',
    'lun': 'luôn', 'ln': 'luôn',
    'nhìu': 'nhiều', 'nhieu': 'nhiều', 'niu': 'nhiều',
    'ít': 'ít', 'jt': 'ít',

    # Động từ phổ biến
    'bik': 'biết', 'bt': 'biết', 'bjết': 'biết',
    'thấy': 'thấy', 'thay': 'thấy', 'thấ': 'thấy',
    'nói': 'nói', 'noj': 'nói', 'nc': 'nói chuyện',
    'học': 'học', 'hok': 'học', 'hoc': 'học',
    'thi': 'thi', 'thj': 'thi',
    'làm': 'làm', 'lam': 'làm', 'lm': 'làm',
    'đi': 'đi', 'dj': 'đi', 'di': 'đi',
    'vào': 'vào', 'vao': 'vào',
    'ra': 'ra',
    'xem': 'xem', 'xê': 'xem',
    'nghe': 'nghe', 'nge': 'nghe',

    # Tính từ phổ biến
    'tốt': 'tốt', 'tot': 'tốt', 'tk': 'tốt',
    'hay': 'hay',
    'đẹp': 'đẹp', 'dep': 'đẹp',
    'dễ': 'dễ', 'de': 'dễ', 'dẽ': 'dễ',
    'khó': 'khó', 'kho': 'khó',
    'ngắn': 'ngắn', 'ngan': 'ngắn',
    'dài': 'dài', 'daj': 'dài',

    # Thời gian
    'h': 'giờ', 'gio': 'giờ', 'hôm': 'hôm', 'hom': 'hôm',
    'hôm nay': 'hôm nay', 'h.nay': 'hôm nay', 'hnay': 'hôm nay',
    'hôm qua': 'hôm qua', 'hqua': 'hôm qua',
    'ngày': 'ngày', 'ngay': 'ngày', 'nay': 'này',

    # Người, em, anh
    'em': 'em', 'e': 'em',
    'anh': 'anh', 'a': 'anh',
    'chị': 'chị', 'chj': 'chị', 'chi': 'chị',
    'cô': 'cô', 'co': 'cô',
    'thầy': 'thầy', 'thay': 'thầy', 'thâ': 'thầy',

    # Cảm thán
    'ok': 'ok', 'oke': 'ok', 'okay': 'ok', 'okela': 'ok',
    'um': 'ừ', 'ừ': 'ừ', 'uh': 'ừ', 'uk': 'ừ',
    'ờ': 'ờ', 'ò': 'ờ',

    # Khác
    'v': 'vậy', 'vậy': 'vậy', 'vay': 'vậy', 'vá': 'vậy', 'z': 'vậy',
    'đâu': 'đâu', 'dau': 'đâu', 'đâ': 'đâu',
    'sao': 'sao',
    'tn': 'thế nào', 'thế nào': 'thế nào', 'thế': 'thế', 'th': 'thế',
    'ntn': 'như thế nào',
    'vs': 'với', 'với': 'với', 'cùng': 'cùng',
    'trong': 'trong', 'tg': 'trong',
    'về': 'về', 've': 'về',
    'cho': 'cho', 'cj': 'cho',
    'nè': 'nè', 'ne': 'nè', 'nì': 'nè',
    'đây': 'đây', 'day': 'đây', 'dâ': 'đây',
    'kia': 'kia', 'kj': 'kia',
    'nhé': 'nhé', 'nhe': 'nhé', 'nhá': 'nhé',
    'ạ': 'ạ', 'a': 'ạ',
    'ha': 'hả', 'hả': 'hả', 'hg': 'hả',
    'chưa': 'chưa', 'chua': 'chưa', 'cga': 'chưa',
    'rồi': 'rồi', 'roi': 'rồi', 'ròj': 'rồi',
}

print(f'Teencode dictionary: {len(TEENCODE_DICT)} mappings')

Teencode dictionary: 178 mappings


### 3.2. Các hàm xử lý

In [6]:
def normalize_teencode(text: str, teencode_dict: Dict[str, str] = None) -> str:
    """
    Chuẩn hóa từ ngữ mạng (teencode) về từ chuẩn tiếng Việt.

    Args:
        text: Input text có thể chứa teencode
        teencode_dict: Dictionary mapping teencode -> từ chuẩn

    Returns:
        Text với teencode đã được thay thế

    Example:
        >>> normalize_teencode("ko bt j cả")
        'không biết gì cả'
    """
    if teencode_dict is None:
        teencode_dict = TEENCODE_DICT

    words = text.split()
    normalized_words = [teencode_dict.get(word, word) for word in words]
    return ' '.join(normalized_words)


def preprocess_vietnamese(text: str, normalize_slang: bool = True) -> str:
    """
    Tiền xử lý văn bản tiếng Việt cho phân loại cảm xúc.

    Các bước:
        1. Chuyển thành chữ thường (lowercase)
        2. Chuẩn hóa teencode/slang (ko → không, j → gì, ...)
        3. Loại bỏ ký tự đặc biệt (giữ lại tiếng Việt có dấu)
        4. Chuẩn hóa khoảng trắng

    Args:
        text: Input text
        normalize_slang: Có chuẩn hóa teencode hay không

    Returns:
        Text đã tiền xử lý

    Example:
        >>> preprocess_vietnamese("Học tập rất tốt! Nhưng thi khó quá.")
        'học tập rất tốt nhưng thi khó quá'
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Normalize teencode/slang
    if normalize_slang:
        text = normalize_teencode(text)

    # 3. Remove special characters (keep Vietnamese diacritics)
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', ' ', text)

    # 4. Normalize whitespace
    return ' '.join(text.split())


print("Functions defined successfully!")

Functions defined successfully!


### 3.3. Test các hàm tiền xử lý

In [7]:
# Test cases
test_samples = [
    "ko bt j cả",
    "cx đc thôi mà",
    "thầy dạy tốt lắm",
    "môn này khó quá hk hiểu j",
    "Học tập rất tốt! Nhưng thi khó quá.",
    "Giảng viên nhiệt tình, bài giảng hay!!!",
    "ko có j đặc biệt",
]

print("=" * 60)
print("TEST TIỀN XỬ LÝ VĂN BẢN")
print("=" * 60)
for sample in test_samples:
    processed = preprocess_vietnamese(sample)
    print(f"Input:  '{sample}'")
    print(f"Output: '{processed}'")
    print("-" * 60)

TEST TIỀN XỬ LÝ VĂN BẢN
Input:  'ko bt j cả'
Output: 'không biết gì cả'
------------------------------------------------------------
Input:  'cx đc thôi mà'
Output: 'cũng được thôi mà'
------------------------------------------------------------
Input:  'thầy dạy tốt lắm'
Output: 'thầy dạy tốt lắm'
------------------------------------------------------------
Input:  'môn này khó quá hk hiểu j'
Output: 'môn này khó quá không hiểu gì'
------------------------------------------------------------
Input:  'Học tập rất tốt! Nhưng thi khó quá.'
Output: 'học tập rất tốt nhưng thi khó quá'
------------------------------------------------------------
Input:  'Giảng viên nhiệt tình, bài giảng hay!!!'
Output: 'giảng viên nhiệt tình bài giảng hay'
------------------------------------------------------------
Input:  'ko có j đặc biệt'
Output: 'không có gì đặc biệt'
------------------------------------------------------------


## 4. Load dữ liệu gốc (Raw Data)

In [8]:
def load_raw_data(raw_dir: str, split: str) -> Tuple[List[str], List[int], List[int]]:
    """
    Load raw UIT-VSFC data.

    Args:
        raw_dir: Path to data/raw directory
        split: One of 'train', 'validation', 'test'

    Returns:
        Tuple of (texts, sentiments, topics)
    """
    split_dir = os.path.join(raw_dir, split)

    with open(os.path.join(split_dir, 'sents.txt'), 'r', encoding='utf-8') as f:
        texts = [line.strip() for line in f.readlines()]

    with open(os.path.join(split_dir, 'sentiments.txt'), 'r', encoding='utf-8') as f:
        sentiments = [int(line.strip()) for line in f.readlines()]

    with open(os.path.join(split_dir, 'topics.txt'), 'r', encoding='utf-8') as f:
        topics = [int(line.strip()) for line in f.readlines()]

    return texts, sentiments, topics


# Load all splits
print("Loading raw data...")
raw_data = {}
for split in config.SPLITS:
    texts, sentiments, topics = load_raw_data(config.RAW_DATA_DIR, split)
    raw_data[split] = {
        'texts': texts,
        'sentiments': sentiments,
        'topics': topics
    }
    print(f"  {split}: {len(texts)} samples")

print(f"\nTotal: {sum(len(d['texts']) for d in raw_data.values())} samples")

Loading raw data...
  train: 11426 samples
  validation: 1583 samples
  test: 3166 samples

Total: 16175 samples


In [9]:
# Xem một số mẫu raw data
print("=" * 60)
print("MẪU DỮ LIỆU GỐC (RAW DATA)")
print("=" * 60)

for split in ['train']:
    print(f"\n--- {split.upper()} ---")
    for i in range(5):
        text = raw_data[split]['texts'][i]
        sentiment = raw_data[split]['sentiments'][i]
        label_name = config.LABEL_MAP[sentiment]
        print(f"{i+1}. [{label_name}] '{text}'")

MẪU DỮ LIỆU GỐC (RAW DATA)

--- TRAIN ---
1. [Positive] 'slide giáo trình đầy đủ .'
2. [Positive] 'nhiệt tình giảng dạy , gần gũi với sinh viên .'
3. [Negative] 'đi học đầy đủ full điểm chuyên cần .'
4. [Negative] 'chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy .'
5. [Positive] 'thầy giảng bài hay , có nhiều bài tập ví dụ ngay trên lớp .'


## 5. Tiền xử lý dữ liệu

In [10]:
def preprocess_split(texts: List[str], normalize_slang: bool = True) -> List[str]:
    """
    Tiền xử lý danh sách văn bản.

    Args:
        texts: List of raw texts
        normalize_slang: Có chuẩn hóa teencode hay không

    Returns:
        List of preprocessed texts
    """
    return [preprocess_vietnamese(text, normalize_slang) for text in texts]


# Process all splits
print("Preprocessing data...")
processed_data = {}

for split in config.SPLITS:
    print(f"  Processing {split}...")
    processed_texts = preprocess_split(raw_data[split]['texts'])
    processed_data[split] = {
        'texts': processed_texts,
        'sentiments': raw_data[split]['sentiments'],
        'topics': raw_data[split]['topics']
    }

print("\nDone!")

Preprocessing data...
  Processing train...
  Processing validation...
  Processing test...

Done!


In [11]:
# So sánh trước/sau khi xử lý
print("=" * 70)
print("SO SÁNH TRƯỚC/SAU KHI TIỀN XỬ LÝ")
print("=" * 70)

np.random.seed(42)
sample_indices = np.random.choice(len(raw_data['train']['texts']), 10, replace=False)

for idx in sample_indices:
    raw_text = raw_data['train']['texts'][idx]
    processed_text = processed_data['train']['texts'][idx]
    sentiment = config.LABEL_MAP[raw_data['train']['sentiments'][idx]]

    print(f"[{sentiment}]")
    print(f"  Raw:        '{raw_text}'")
    print(f"  Processed:  '{processed_text}'")

    # Highlight changes
    if raw_text.lower() != processed_text:
        print(f"  >>> CHANGED")
    print("-" * 70)

SO SÁNH TRƯỚC/SAU KHI TIỀN XỬ LÝ
[Positive]
  Raw:        'có khả năng truyền đạt tốt .'
  Processed:  'có khả năng truyền đạt tốt'
  >>> CHANGED
----------------------------------------------------------------------
[Negative]
  Raw:        'khi có nhóm lên seminar thì phải có ít nhất ba câu hỏi thì mới " đậu " , không cần biết câu hỏi là gì , không đủ ba câu thì phải làm lại .'
  Processed:  'khi có nhóm lên seminar thì phải có ít nhất ba câu hỏi thì mới đậu không cần biết câu hỏi là gì không đủ ba câu thì phải làm lại'
  >>> CHANGED
----------------------------------------------------------------------
[Negative]
  Raw:        'thầy giảng buồn ngủ , không nhiệt tình , bài tập thực hành không có .'
  Processed:  'thầy giảng buồn ngủ không nhiệt tình bài tập thực hành không có'
  >>> CHANGED
----------------------------------------------------------------------
[Positive]
  Raw:        'sữa lỗi cho sinh viên .'
  Processed:  'sữa lỗi cho sinh viên'
  >>> CHANGED
----------------------

## 6. Thống kê sau tiền xử lý

In [12]:
# Thống kê cơ bản
print("=" * 60)
print("THỐNG KÊ DỮ LIỆU SAU TIỀN XỬ LÝ")
print("=" * 60)

for split in config.SPLITS:
    texts = processed_data[split]['texts']
    sentiments = processed_data[split]['sentiments']

    # Length stats
    lengths = [len(t.split()) for t in texts]

    # Label distribution
    label_counts = Counter(sentiments)

    print(f"\n--- {split.upper()} ---")
    print(f"  Samples: {len(texts)}")
    print(f"  Word count: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}")
    print(f"  Label distribution:")
    for label in range(config.NUM_CLASSES):
        count = label_counts.get(label, 0)
        pct = count / len(texts) * 100
        print(f"    {config.LABEL_MAP[label]}: {count} ({pct:.1f}%)")

THỐNG KÊ DỮ LIỆU SAU TIỀN XỬ LÝ

--- TRAIN ---
  Samples: 11426
  Word count: min=1, max=147, mean=12.7
  Label distribution:
    Negative: 5325 (46.6%)
    Neutral: 458 (4.0%)
    Positive: 5643 (49.4%)

--- VALIDATION ---
  Samples: 1583
  Word count: min=1, max=150, mean=12.0
  Label distribution:
    Negative: 705 (44.5%)
    Neutral: 73 (4.6%)
    Positive: 805 (50.9%)

--- TEST ---
  Samples: 3166
  Word count: min=1, max=93, mean=12.6
  Label distribution:
    Negative: 1409 (44.5%)
    Neutral: 167 (5.3%)
    Positive: 1590 (50.2%)


In [13]:
# Đếm số lượng sample bị thay đổi
print("=" * 60)
print("THỐNG KÊ THAY ĐỔI")
print("=" * 60)

for split in config.SPLITS:
    raw_texts = raw_data[split]['texts']
    processed_texts = processed_data[split]['texts']

    changed = sum(1 for r, p in zip(raw_texts, processed_texts) if r.lower() != p)
    print(f"{split}: {changed}/{len(raw_texts)} samples changed ({changed/len(raw_texts)*100:.1f}%)")

THỐNG KÊ THAY ĐỔI
train: 11426/11426 samples changed (100.0%)
validation: 1583/1583 samples changed (100.0%)
test: 3166/3166 samples changed (100.0%)


## 7. Lưu dữ liệu đã xử lý

In [14]:
def save_processed_data(data: Dict, output_dir: str, split: str):
    """
    Lưu dữ liệu đã xử lý vào thư mục output.

    Args:
        data: Dict containing 'texts', 'sentiments', 'topics'
        output_dir: Output directory path
        split: Split name (train/validation/test)
    """
    split_dir = os.path.join(output_dir, split)
    os.makedirs(split_dir, exist_ok=True)

    # Save texts
    with open(os.path.join(split_dir, 'sents.txt'), 'w', encoding='utf-8') as f:
        for text in data['texts']:
            f.write(text + '\n')

    # Save sentiments
    with open(os.path.join(split_dir, 'sentiments.txt'), 'w', encoding='utf-8') as f:
        for sentiment in data['sentiments']:
            f.write(str(sentiment) + '\n')

    # Save topics
    with open(os.path.join(split_dir, 'topics.txt'), 'w', encoding='utf-8') as f:
        for topic in data['topics']:
            f.write(str(topic) + '\n')

    print(f"  Saved {len(data['texts'])} samples to {split_dir}")


# Save all splits
print("Saving processed data...")
print(f"Output directory: {config.PROCESSED_DATA_DIR}")
print()

for split in config.SPLITS:
    save_processed_data(processed_data[split], config.PROCESSED_DATA_DIR, split)

print("\nDone! Data saved successfully.")

Saving processed data...
Output directory: /content/drive/MyDrive/Student-Feedback-Sentiment-Analysis/data/processed

  Saved 11426 samples to /content/drive/MyDrive/Student-Feedback-Sentiment-Analysis/data/processed/train
  Saved 1583 samples to /content/drive/MyDrive/Student-Feedback-Sentiment-Analysis/data/processed/validation
  Saved 3166 samples to /content/drive/MyDrive/Student-Feedback-Sentiment-Analysis/data/processed/test

Done! Data saved successfully.


In [15]:
# Verify saved data
print("=" * 60)
print("KIỂM TRA DỮ LIỆU ĐÃ LƯU")
print("=" * 60)

for split in config.SPLITS:
    split_dir = os.path.join(config.PROCESSED_DATA_DIR, split)

    # Count lines in each file
    with open(os.path.join(split_dir, 'sents.txt'), 'r', encoding='utf-8') as f:
        sents_count = len(f.readlines())
    with open(os.path.join(split_dir, 'sentiments.txt'), 'r', encoding='utf-8') as f:
        sentiments_count = len(f.readlines())
    with open(os.path.join(split_dir, 'topics.txt'), 'r', encoding='utf-8') as f:
        topics_count = len(f.readlines())

    print(f"{split}: sents={sents_count}, sentiments={sentiments_count}, topics={topics_count}")
    assert sents_count == sentiments_count == topics_count, "Count mismatch!"

print("\nAll files verified successfully!")

KIỂM TRA DỮ LIỆU ĐÃ LƯU
train: sents=11426, sentiments=11426, topics=11426
validation: sents=1583, sentiments=1583, topics=1583
test: sents=3166, sentiments=3166, topics=3166

All files verified successfully!


## 8. Tóm tắt

### Các bước đã thực hiện:

1. **Load dữ liệu gốc** từ `data/raw/`
2. **Tiền xử lý văn bản**:
   - Chuyển thành chữ thường
   - Chuẩn hóa teencode (ko → không, j → gì, cx → cũng, ...)
   - Loại bỏ ký tự đặc biệt
   - Chuẩn hóa khoảng trắng
3. **Lưu dữ liệu đã xử lý** vào `data/processed/`

### Output:

```
data/processed/
├── train/
│   ├── sents.txt      (11,426 samples)
│   ├── sentiments.txt
│   └── topics.txt
├── validation/
│   ├── sents.txt      (1,583 samples)
│   ├── sentiments.txt
│   └── topics.txt
└── test/
    ├── sents.txt      (3,166 samples)
    ├── sentiments.txt
    └── topics.txt
```

### Bước tiếp theo:
- **Giai đoạn 2**: Trích xuất đặc trưng (PhoBERT embeddings, TF-IDF, SentiWordNet)
- **Giai đoạn 3**: Xây dựng mô hình

In [16]:
# Quick test: Load processed data using src/data_utils.py
import sys
sys.path.append(config.BASE_DIR)

from src.data_utils import load_data, LABEL_MAP

print("Testing load_data from src/data_utils.py...")
texts, labels = load_data(config.PROCESSED_DATA_DIR, 'train')
print(f"Loaded {len(texts)} training samples")
print(f"Sample: '{texts[0]}' -> Label: {labels[0]} ({LABEL_MAP[labels[0]]})")
print("\nData pipeline ready for model training!")

Testing load_data from src/data_utils.py...
Loaded 11426 training samples
Sample: 'slide giáo trình đầy đủ' -> Label: 2 (Positive)

Data pipeline ready for model training!
